In [ ]:
# STEP 1: Install required libraries
!pip install torch torchvision torchaudio transformers accelerate bitsandbytes huggingface_hub python-dotenv --quiet

In [ ]:
# STEP 2: Load Hugging Face token from .env and login
from dotenv import load_dotenv
import os
from huggingface_hub import login

load_dotenv()  # loads variables from a .env file in the notebook's working directory
hf_token = os.environ.get("HUGGINGFACE_TOKEN")
if not hf_token:
    raise ValueError("HUGGINGFACE_TOKEN not found. Create a .env file with: HUGGINGFACE_TOKEN=your_token")
# Optionally add .env to .gitignore to avoid committing secrets
login(token=hf_token)

In [ ]:
# STEP 3: Load Mistral 7B with 4-bit quantization
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

model_name = "mistralai/Mistral-7B-Instruct-v0.1"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

print("Loading model (this may take a while)...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Model loaded. Device:", device)

In [ ]:
# STEP 4: Create a text-generation pipeline
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

In [ ]:
# STEP 5: Prompt the model
prompt = """
Design a star schema for a sales analytics data warehouse.
Include SQL DDL statements to create the tables: sales_fact, product_dim, customer_dim, time_dim, and store_dim.
Use PostgreSQL syntax.
"""

# STEP 6: Generate and Print the Output
output = generator(prompt, max_new_tokens=512, temperature=0.5, do_sample=True)
print(" Star Schema SQL Output:\n")
print(output[0]['generated_text'])

# ELI5 — What this notebook does (plain English)

- Install: installs required Python libraries like `transformers` and `bitsandbytes`.
- .env & login: loads your Hugging Face token from a `.env` file using `python-dotenv` and logs in so the notebook can download the model.
- Model load: downloads and loads Mistral-7B using 4-bit quantization (smaller model size, lower GPU memory).
- Pipeline: creates a text-generation `pipeline` for convenient inference.
- Prompt & generate: sends a prompt asking for a star schema and prints the model's SQL DDL output.

Extra notes:
- 4-bit quantization reduces memory by storing weights in 4 bits (not full precision). This lets large models fit on smaller GPUs but may slightly reduce output quality.
- `device_map="auto"` lets the `transformers` library place model layers across available devices to avoid out-of-memory errors.
- Keep your `.env` secret: add it to `.gitignore`.